<a href="https://colab.research.google.com/github/chadallen/insider_trading_detection/blob/main/Detect_insider_trading_on_prediction_markets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Polymarket Insider Trading Detector

**Links:** [GitHub](https://github.com/chadallen/insider_trading_detection) · [Google Drive (data)](https://drive.google.com/drive/folders/1rD3SXjFoLfCmlQOeftVj1LI-pbEu1o1u?usp=sharing) · [Dashboard](https://polymarket-dashboard-roan.vercel.app/)

---

## What this does

This notebook looks for suspicious trading patterns in resolved Polymarket prediction markets using two independent signals:

**1. Price-based scoring** (free — uses Polymarket's public API)
- VPIN — measures order flow imbalance (adapted from Easley, López de Prado & O'Hara, 2011)
- Time-weighted VPIN — same, but weighted toward moves closer to resolution
- Resolution surprise — how wrong was the market's implied probability vs. what actually happened?
- Late move ratio — what % of total price movement happened right before resolution?
- All four signals are fed into an **Isolation Forest** (unsupervised anomaly detector) to rank markets

**2. Wallet-based scoring** (costs Dune credits — run selectively)
- New wallet ratio — did fresh wallets appear right before resolution?
- Burst score — did any wallet place a suspicious number of trades in a short window?
- Directional consensus — was trading overwhelmingly one-sided?
- Trade VPIN — real order flow imbalance from actual on-chain trade data

The two scores are averaged into a **combined suspicion score** and pushed to GitHub, where the dashboard reads them live.

**Limitations:**
- Polymarket only gives 12h price granularity for closed markets — VPIN is approximated from price movement, not actual order flow
- Political markets have no price history via CLOB API (data retention wall) — wallet scoring only
- ~130 markets is a small sample; model is unsupervised (no labeled training data yet)

---

## Setup (first time / forked notebook)

1. **Update Cell 0** with your GitHub repo and Drive folder name
2. **Add Colab Secrets** (click the key icon in the left sidebar):
   - `DUNE_API_KEY` — from [dune.com](https://dune.com) (free tier works)
   - `GITHUB_TOKEN` — GitHub personal access token with `repo` scope
3. Make sure your GitHub repo has an `outputs/` folder

---

## Run order guide

### Normal session (data already saved to Drive)
If Cell 2 shows all checkpoints as loaded, you can skip the expensive re-fetches:
```
Cell 0 → Cell 1 → Cell 2 (load Drive)
→ Cell 6 (re-score with Isolation Forest)
→ Cell 11 (define wallet scoring rules)
→ Cell 14 (combine scores)
→ Cell 15 (save) → Cell 16 (push to GitHub)
```
⏱ ~2 min · 0 Dune credits

### Refresh price scores (free)
Run when you want to pick up newly resolved markets:
```
Cell 0 → Cell 1 → Cell 2
→ Cell 3 (fetch markets) → Cell 4 (fetch price histories, ~5 min)
→ Cell 5 (compute features) → Cell 6 (Isolation Forest)
→ Cell 11 → Cell 13 (re-run Dune for new top 15) → Cell 14
→ Cell 15 → Cell 16
```
⏱ ~10 min · ~4 Dune credits (Cell 13 only)

### Full refresh (run sparingly)
Run when adding new labeled cases or Drive is empty:
```
All cells in order: Cell 0 through Cell 16
```
⏱ ~20 min · ~8 Dune credits

### When to run the expensive Dune cells

| Situation | Cell 9 (labeled markets) | Cell 13 (top 15) |
|-----------|:---:|:---:|
| Added new labeled cases to Cell 8 | ✅ Yes | — |
| Top 15 changed after price refresh | — | ✅ Yes |
| Just refreshing the dashboard | — | ✅ Yes |
| Drive pkl files missing | ✅ Yes | ✅ Yes |
| Routine weekly refresh | — | ✅ Yes |

**Dune free tier:** ~20–30 credits/month. Each query costs ~4 credits.


In [26]:
#@title Cell 0 — User Configuration
# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS CELL BEFORE RUNNING ANYTHING ELSE                   ║
# ║  If you forked this notebook, update the values below.         ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── GitHub settings ───────────────────────────────────────────────
# Format: "username/repo-name"
# Your repo needs an outputs/ folder (create one with a placeholder file)
GITHUB_REPO   = "chadallen/insider_trading_detection"
GITHUB_BRANCH = "main"

# ── Google Drive settings ──────────────────────────────────────────
# Folder name in your Google Drive root for pkl checkpoints
DRIVE_FOLDER  = "insider_trading_poc"

# ── API keys ───────────────────────────────────────────────────────
# Do NOT paste keys here. Add them as Colab Secrets instead:
#   1. Click the key icon in the left sidebar
#   2. Add secret: DUNE_API_KEY   (from dune.com, free tier works)
#   3. Add secret: GITHUB_TOKEN   (github.com/settings/tokens, repo scope)

# ── Data settings ─────────────────────────────────────────────────
DAYS_BACK      = 548   # how far back to look for resolved markets
TOP_N_MARKETS  = 15    # how many markets to fetch wallet data for in Cell 13

print("✅ Configuration loaded")
print(f"   GitHub repo:   {GITHUB_REPO}")
print(f"   Drive folder:  {DRIVE_FOLDER}")
print(f"   Days back:     {DAYS_BACK}")
print(f"   Top N markets: {TOP_N_MARKETS}")

✅ Configuration loaded
   GitHub repo:   chadallen/insider_trading_detection
   Drive folder:  insider_trading_poc
   Days back:     548
   Top N markets: 15


In [27]:
#@title Cell 1 — Install dependencies and imports
# Run once per session. All core imports live here so any cell can be run independently.

!pip install requests pandas numpy scikit-learn plotly -q
!pip install PyGithub -q

import requests
import pandas as pd
import numpy as np
import json
import os
import pickle
import time
import io
from datetime import datetime, timedelta, timezone
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print("✅ Dependencies installed and imports loaded")

✅ Dependencies installed and imports loaded


In [20]:
#@title Cell 2 — Mount Google Drive and load saved data
# Loads pkl checkpoints so you can skip expensive re-fetches.
# Variables that load as None will be recomputed when their cell runs.

from google.colab import drive
import os, pickle

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive already mounted ✅")

SAVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}\n")

def load_checkpoint(name):
    path = f"{SAVE_DIR}/{name}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  ✅ Loaded {name}")
        return data
    else:
        print(f"  ⚠️  Not found: {name} — will be computed when its cell runs")
        return None

print("Loading saved data...\n")
df_markets    = load_checkpoint("df_markets")
df_scored     = load_checkpoint("df_scored")
histories     = load_checkpoint("histories")
dune_results  = load_checkpoint("dune_results")
df_wallet_agg = load_checkpoint("df_wallet_agg")
df_combined   = load_checkpoint("df_combined")

print("\nDone.")
print("If all loaded ✅, you can skip to Cell 6 (re-score) or Cell 13 (refresh top 15 wallet data).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save directory: /content/drive/MyDrive/insider_trading_poc

Loading saved data...

  ✅ Loaded df_markets
  ✅ Loaded df_scored
  ✅ Loaded histories
  ✅ Loaded dune_results
  ✅ Loaded df_wallet_agg
  ✅ Loaded df_combined

Done.
If all loaded ✅, you can skip to Cell 6 (re-score) or Cell 13 (refresh top 15 wallet data).


---
## Section A: Price-Based Scoring

**Cells 3–6.** Uses Polymarket's free public API. No API keys required.

**Skip to Cell 6** if `df_scored` loaded from Drive — it will re-run the Isolation Forest on the saved features.

**Run Cells 3–6** if:
- `df_scored` is None (not found on Drive)
- You want to pick up newly resolved markets
- It has been more than a week since your last run

> Cell 4 fetches price history for ~130 markets and takes 3–5 minutes.

In [ ]:
#@title Cell 3 — Fetch resolved markets
# Calls Gamma API to get recently resolved markets with token IDs.
# Result stored in df_markets (~130–180 markets).
# ⏱ ~10 seconds · 0 credits

import requests
import pandas as pd
import json
from datetime import datetime, timedelta, timezone

def fetch_resolved_markets(days_back=365, limit=200):
    end_date   = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=days_back)
    url    = "https://gamma-api.polymarket.com/markets"
    params = {
        "closed": "true", "limit": limit,
        "after": start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "order": "endDate", "ascending": "false",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    markets = []
    for m in response.json():
        raw_tokens = m.get("clobTokenIds", "[]")
        try:
            tokens   = json.loads(raw_tokens) if isinstance(raw_tokens, str) else raw_tokens
            token_id = tokens[0] if tokens else None
        except:
            token_id = None
        markets.append({
            "market_id":       m.get("conditionId"),
            "token_id":        token_id,
            "question":        m.get("question"),
            "resolution_time": m.get("endDate"),
            "final_price":     m.get("outcomePrices"),
            "volume":          float(m.get("volume") or 0),
        })
    df = pd.DataFrame(markets)
    df = df[df["token_id"].notna() & (df["volume"] > 0)]
    print(f"Fetched {len(df)} markets with trade data")
    return df

df_markets = fetch_resolved_markets(days_back=DAYS_BACK)
print(df_markets[["question", "volume"]].head(5).to_string())

Fetched 182 markets with trade data
                                          question         volume
1   Espresso FDV above $500M one day after launch?   56135.941632
3   Espresso FDV above $400M one day after launch?   90657.051213
6    Espresso FDV above $50M one day after launch?   83280.457952
11  Espresso FDV above $300M one day after launch?  140357.762760
15  Espresso FDV above $100M one day after launch?  211099.127702


In [ ]:
#@title Cell 4 — Fetch price histories
# Calls CLOB API for each market's price history.
# Closed markets only support 12h+ granularity (fidelity=720 minimum).
# Only keeps markets with starting price 0.15–0.85 (otherwise not contested).
# ⏱ 3–5 minutes · 0 credits

def fetch_price_history(token_id, resolution_time, hours_before=48):
    if isinstance(resolution_time, str):
        res_time = datetime.fromisoformat(resolution_time.replace("Z", "+00:00"))
    else:
        res_time = resolution_time
    start_time = res_time - timedelta(hours=hours_before)
    url    = "https://clob.polymarket.com/prices-history"  # Note: clob not gamma-api
    params = {
        "market":   token_id,
        "interval": "max",
        "fidelity": 720,  # 12h min for closed markets — undocumented limitation
        "startTs":  int(start_time.timestamp()),
        "endTs":    int(res_time.timestamp()),
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        if not data or "history" not in data or not data["history"]:
            return pd.DataFrame()
        df = pd.DataFrame(data["history"])
        df = df.rename(columns={"t": "timestamp", "p": "price"})  # API uses t/p not timestamp/price
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        df["price"]     = df["price"].astype(float)
        return df
    except Exception:
        return pd.DataFrame()

print(f"Fetching price histories for {len(df_markets)} markets...")
histories = {}
for i, (_, row) in enumerate(df_markets.iterrows()):
    if i % 25 == 0: print(f"  {i}/{len(df_markets)}...")
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    if len(history) < 3: continue
    if not (0.15 <= history["price"].iloc[0] <= 0.85): continue
    histories[row["token_id"]] = history

print(f"\n✅ Cached price histories for {len(histories)} markets")

Fetching price histories for 182 markets...
  0/182...
  25/182...
  50/182...
  75/182...
  100/182...
  125/182...
  150/182...
  175/182...

✅ Cached price histories for 124 markets


In [ ]:
#@title Cell 5 — Compute price features
# Computes VPIN, time-weighted VPIN, resolution surprise, late move ratio,
# price volatility, max single move, and total price move.
# Results stored in df_scored.
# ⏱ ~5 seconds · 0 credits

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def compute_vpin_from_prices(history_df):
    if len(history_df) < 3: return None
    prices = history_df["price"].values
    changes = np.diff(prices)
    buy  = np.where(changes > 0, changes, 0).sum()
    sell = np.where(changes < 0, -changes, 0).sum()
    total = buy + sell
    return abs(buy - sell) / total if total > 0 else 0.0

def compute_time_weighted_vpin(history_df):
    """VPIN with linear weights — moves closer to resolution count more."""
    if len(history_df) < 3: return None
    prices  = history_df["price"].values
    changes = np.diff(prices)
    n       = len(changes)
    weights = np.arange(1, n + 1, dtype=float)  # 1, 2, 3...N
    w_buy   = np.where(changes > 0, changes * weights, 0).sum()
    w_sell  = np.where(changes < 0, -changes * weights, 0).sum()
    total   = w_buy + w_sell
    return abs(w_buy - w_sell) / total if total > 0 else 0.0

def compute_price_features(history_df):
    if len(history_df) < 2: return None
    prices  = history_df["price"].values
    changes = np.abs(np.diff(prices))
    return {
        "price_volatility": changes.std() if len(changes) > 1 else 0,
        "max_single_move":  changes.max() if len(changes) > 0 else 0,
        "final_price":      prices[-1],
        "starting_price":   prices[0],
        "total_price_move": abs(prices[-1] - prices[0]),
    }

def compute_resolution_surprise(history_df):
    if len(history_df) < 2: return None, None
    prices           = history_df["price"].values
    actual           = 1.0 if prices[-1] > 0.5 else 0.0
    surprise         = abs(actual - prices[0])
    total_move       = abs(prices[-1] - prices[0])
    final_step       = abs(prices[-1] - prices[-2])
    late_move_ratio  = (final_step / total_move) if total_move > 0.01 else 0.0
    return surprise, late_move_ratio

results = []
for _, row in df_markets.iterrows():
    history = histories.get(row["token_id"])
    if history is None or len(history) < 3: continue
    vpin      = compute_vpin_from_prices(history)
    tw_vpin   = compute_time_weighted_vpin(history)
    features  = compute_price_features(history)
    surprise, late_move = compute_resolution_surprise(history)
    if any(v is None for v in [vpin, tw_vpin, features, surprise, late_move]): continue
    results.append({
        "question":           row["question"],
        "volume":             row["volume"],
        "vpin_score":         vpin,
        "time_weighted_vpin": tw_vpin,
        "surprise_score":     surprise,
        "late_move_ratio":    late_move,
        **features,
    })

df_scored = pd.DataFrame(results)
print(f"✅ Computed features for {len(df_scored)} markets")
print(df_scored[["question", "vpin_score", "surprise_score", "late_move_ratio"]].head(5).to_string())

✅ Computed features for 124 markets
                                            question  vpin_score  surprise_score  late_move_ratio
0     Espresso FDV above $400M one day after launch?    0.725652           0.265         0.011342
1     Espresso FDV above $300M one day after launch?    0.905571           0.480         0.013556
2     Espresso FDV above $200M one day after launch?    0.465875           0.550         0.845314
3  Will Paradex launch a token by February 28, 2026?    0.114844           0.585         0.000855
4     Sentient FDV above $1.5B one day after launch?    0.449275           0.240         0.245161


In [ ]:
#@title Cell 6 — Score markets with Isolation Forest
# Runs unsupervised anomaly detection across 8 price features.
# contamination=0.1 → ~10% of markets flagged as anomalous.
# Adds 'suspicion_score' to df_scored (higher = more suspicious).
# Can be re-run on loaded data without re-fetching price histories.
# ⏱ ~2 seconds · 0 credits

feature_cols = [
    "vpin_score",         # order flow imbalance (price proxy)
    "time_weighted_vpin", # same, weighted toward resolution
    "volume",             # total USD traded
    "total_price_move",   # overall price change start→end
    "price_volatility",   # std dev of 12h price changes
    "max_single_move",    # largest single 12h price jump
    "surprise_score",     # how wrong was the market before resolution
    "late_move_ratio",    # % of total move that happened right before resolution
]

X        = df_scored[feature_cols].fillna(0)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
model    = IsolationForest(contamination=0.1, random_state=42)

df_scored["anomaly_score"]   = model.fit_predict(X_scaled)
df_scored["suspicion_score"] = -model.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "surprise_score", "late_move_ratio", "volume"]
].reset_index(drop=True)

print(f"✅ Scored {len(df_scored)} markets\n")
print("Top 15 by price-based suspicion score:")
print(top.to_string())

✅ Scored 124 markets

Top 15 by price-based suspicion score:
                                               question  suspicion_score  vpin_score  surprise_score  late_move_ratio        volume
0        Sentient FDV above $800M one day after launch?         0.118841    0.026975           0.305         3.785714  4.498054e+05
1     Will Almanak FDV above $50M one day after launch?         0.110967    0.997650           0.850         0.000589  2.053399e+05
2   Over $500K committed to the Sol Raffle public sale?         0.084705    0.988258           0.765         0.005941  4.512802e+04
3         Over $20M committed to the Space public sale?         0.078781    0.365152           0.800         0.724828  2.327351e+05
4        Hyperlend FDV above $20M one day after launch?         0.063232    0.157895           0.190         1.000000  1.482730e+06
5          Rainbow FDV above $30M one day after launch?         0.048276    1.000000           0.425         0.811543  1.310044e+05
6             T

---
## Section B: Wallet-Based Scoring (Dune Analytics)

**Cells 7–14.** Queries on-chain trade data from `polymarket_polygon.market_trades`.

**Dune credits used:** ~4 per query. Free tier = ~20–30 credits/month.

| Cell | What it does | Credits | When to run |
|------|-------------|---------|-------------|
| Cell 7 | Define Dune API helpers | 0 | Always — just defines functions |
| Cell 8 | Load labeled cases | 0 | Always |
| Cell 9 | Fetch raw trades for labeled markets | ~4 | Only when adding new labeled cases |
| Cell 10 | Compute wallet features from raw data | 0 | After Cell 9 |
| Cell 11 | Define wallet suspicion scoring rules | 0 | Always — just defines a function |
| Cell 12 | Score labeled markets (validation) | 0 | After Cells 10 + 11 |
| **Cell 13** | **Aggregated query for top N markets** | **~4** | **When top 15 has changed** |
| Cell 14 | Combine price + wallet scores | 0 | After Cell 13 |

**If `dune_results` and `df_wallet_agg` loaded from Drive ✅:** Run Cell 7 (define helpers), Cell 11 (define rules), then jump to Cell 14.

In [7]:
#@title Cell 7 — Dune API helper functions
# Defines execute_sql(), poll_until_done(), fetch_results().
# Required by Cell 9 and Cell 13 — always run this cell first.
# ⏱ Instant · 0 credits

import time
from google.colab import userdata

DUNE_API_KEY = userdata.get('DUNE_API_KEY')
DUNE_HEADERS = {"X-Dune-Api-Key": DUNE_API_KEY, "Content-Type": "application/json"}

def execute_sql(sql):
    """Submit a Dune SQL query, return the execution_id."""
    r = requests.post(
        "https://api.dune.com/api/v1/sql/execute",
        headers=DUNE_HEADERS, json={"sql": sql, "performance": "medium"}
    )
    r.raise_for_status()
    return r.json()["execution_id"]

def poll_until_done(execution_id, timeout=180, interval=5):
    """Poll until a Dune query completes or times out."""
    start = time.time()
    while time.time() - start < timeout:
        r = requests.get(
            f"https://api.dune.com/api/v1/execution/{execution_id}/status",
            headers=DUNE_HEADERS
        )
        r.raise_for_status()
        state = r.json()["state"]
        if state == "QUERY_STATE_COMPLETED": return True
        if state in ("QUERY_STATE_FAILED", "QUERY_STATE_CANCELLED"):
            print(f"  Query failed: {state}")
            return False
        print(f"  Status: {state} — waiting {interval}s...")
        time.sleep(interval)
    print("  Timed out.")
    return False

def fetch_results(execution_id):
    """Fetch results from a completed Dune execution."""
    r = requests.get(
        f"https://api.dune.com/api/v1/execution/{execution_id}/results",
        headers=DUNE_HEADERS
    )
    r.raise_for_status()
    rows = r.json().get("result", {}).get("rows", [])
    return pd.DataFrame(rows)

print("✅ Dune helpers defined: execute_sql(), poll_until_done(), fetch_results()")
print("   Required by Cell 9 and Cell 13.")

✅ Dune helpers defined: execute_sql(), poll_until_done(), fetch_results()
   Required by Cell 9 and Cell 13.


---
## Section C: Labeled Data & Validation

**Cells 8–12.** Ground truth cases used to calibrate the wallet scoring model.

> **Note on missing price history:** Political markets have no CLOB price data due to Polymarket's data retention policy (CLOB only retains price data for a limited window after resolution). All scoring for labeled cases is therefore wallet-only.

In [ ]:
#@title Cell 8 — Load labeled insider trading cases
# Hardcoded ground truth cases. Add new confirmed/suspected cases here.
# Labels: CONFIRMED = reported by credible source · SUSPECTED = anomalous, unconfirmed · POSSIBLE = not investigated
# ⏱ Instant · 0 credits

import io

LABELS_CSV = """market_question,label
"Will the US strike Iran by February 28, 2026?",CONFIRMED
"Khamenei out as Supreme Leader of Iran by March 31, 2026?",SUSPECTED
"Will Nicolás Maduro be removed from office by January 31, 2026?",SUSPECTED
"Will the US take military action against Venezuela by January 31, 2026?",SUSPECTED
"Will María Corina Machado win the 2025 Nobel Peace Prize?",CONFIRMED
"Will Israel strike Iran between June 12-18, 2025?",CONFIRMED
"Which crypto company will ZachXBT expose for insider trading?",SUSPECTED
"Will Taylor Swift and Travis Kelce get engaged in 2025?",SUSPECTED
"Who will be the most searched person on Google in 2025?",SUSPECTED
"Will Google release Gemini 3.0 Flash on specific date window?",POSSIBLE
"Will the 2024 US presidential election be won by Donald Trump?",POSSIBLE
"""

df_labels = pd.read_csv(io.StringIO(LABELS_CSV))
print(f"Loaded {len(df_labels)} labeled cases")
print(df_labels[["market_question", "label"]].to_string())

Loaded 11 labeled cases
                                                            market_question      label
0                             Will the US strike Iran by February 28, 2026?  CONFIRMED
1                 Khamenei out as Supreme Leader of Iran by March 31, 2026?  SUSPECTED
2           Will Nicolás Maduro be removed from office by January 31, 2026?  SUSPECTED
3   Will the US take military action against Venezuela by January 31, 2026?  SUSPECTED
4                 Will María Corina Machado win the 2025 Nobel Peace Prize?  CONFIRMED
5                         Will Israel strike Iran between June 12-18, 2025?  CONFIRMED
6             Which crypto company will ZachXBT expose for insider trading?  SUSPECTED
7                   Will Taylor Swift and Travis Kelce get engaged in 2025?  SUSPECTED
8                   Who will be the most searched person on Google in 2025?  SUSPECTED
9             Will Google release Gemini 3.0 Flash on specific date window?   POSSIBLE
10           Will t

In [21]:
#@title Cell 9a — Fetch wallet features for labeled markets from saved PKL [CHEAP]


# Run this instead of re-running Cell 9 if you have an updated PKL file

# Load the full dune_results from Drive (has all 4 markets)
with open(f"{SAVE_DIR}/dune_results.pkl", "rb") as f:
    dune_results = pickle.load(f)

print("Loaded from Drive:")
for name, df in dune_results.items():
    print(f"  {name}: {len(df)} row(s)")

Loaded from Drive:
  maduro: 0 row(s)
  zachxbt: 0 row(s)
  nobel: 0 row(s)
  iran: 0 row(s)


In [28]:
#@title Cell 9b — Fetch wallet features for labeled markets from Dune [EXPENSIVE]
# Uses pre-aggregated SQL (1 row per market) to stay within Dune free tier limits.
# Results stored in dune_results as a dict of DataFrames (one per market).
#
# COSTS ~4 DUNE CREDITS. Skip if dune_results loaded from Drive ✅
# Re-run only when adding new labeled cases to Cell 8.

LABELED_MARKET_CONFIGS = {
    "maduro": {
        "label": "SUSPECTED",
        "question_filter": "question LIKE '%Maduro%'",
        "start": "2026-01-28", "end": "2026-01-31",
        "resolution": "2026-01-31T00:00:00",
    },
    "zachxbt": {
        "label": "SUSPECTED",
        "question_filter": "question = 'Will Axiom be accused of insider trading?'",
        "start": "2026-02-24", "end": "2026-02-27",
        "resolution": "2026-02-26T23:59:00",
    },
    "nobel": {
        "label": "CONFIRMED",
        "question_filter": "question = 'Will María Corina Machado win the Nobel Peace Prize in 2025?'",
        "start": "2025-10-08", "end": "2025-10-11",
        "resolution": "2025-10-10T11:00:00",
    },
    "iran": {
        "label": "CONFIRMED",
        "question_filter": "question = 'US strikes Iran by February 28, 2026?'",
        "start": "2026-02-26", "end": "2026-02-28",
        "resolution": "2026-02-28T23:59:00",
    },
}

RESOLUTION_TIMES = {k: v["resolution"] for k, v in LABELED_MARKET_CONFIGS.items()}

def build_labeled_sql(question_filter, start, end, resolution):
    resolution_trino = resolution.replace("T", " ")
    return f"""
WITH base AS (
    SELECT block_time, maker, price, amount, token_outcome_name
    FROM polymarket_polygon.market_trades
    WHERE {question_filter}
      AND block_time BETWEEN TIMESTAMP '{start}' AND TIMESTAMP '{end}'
),
totals AS (
    SELECT
        COUNT(*)              AS trade_count,
        COUNT(DISTINCT maker) AS unique_wallets,
        SUM(amount)           AS total_volume,
        ABS(
            SUM(CASE WHEN price > 0.5  THEN amount ELSE 0 END) -
            SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END)
        ) / NULLIF(SUM(amount), 0) AS trade_vpin
    FROM base
),
wallet_vols AS (
    SELECT maker, SUM(amount) AS wv FROM base GROUP BY maker
),
top3 AS (
    SELECT SUM(wv) AS top3_volume
    FROM (
        SELECT wv, ROW_NUMBER() OVER (ORDER BY wv DESC) AS rn
        FROM wallet_vols
    ) ranked
    WHERE rn <= 3
),
first_seen AS (
    SELECT maker, MIN(block_time) AS first_trade FROM base GROUP BY maker
),
new_12h AS (
    SELECT COALESCE(SUM(b.amount), 0) AS vol_12h
    FROM base b
    JOIN first_seen f ON b.maker = f.maker
    WHERE f.first_trade >= TIMESTAMP '{resolution_trino}' - INTERVAL '12' HOUR
),
new_6h AS (
    SELECT COALESCE(SUM(b.amount), 0) AS vol_6h
    FROM base b
    JOIN first_seen f ON b.maker = f.maker
    WHERE f.first_trade >= TIMESTAMP '{resolution_trino}' - INTERVAL '6' HOUR
),
burst AS (
    SELECT MAX(n) AS burst_score
    FROM (
        SELECT maker, DATE_TRUNC('hour', block_time) AS h, COUNT(*) AS n
        FROM base GROUP BY maker, DATE_TRUNC('hour', block_time)
    ) x
),
directional AS (
    SELECT MAX(ov) * 1.0 / NULLIF(SUM(ov), 0) AS directional_consensus
    FROM (
        SELECT token_outcome_name, SUM(amount) AS ov
        FROM base GROUP BY token_outcome_name
    ) x
)
SELECT
    t.trade_count,
    t.unique_wallets,
    t.total_volume,
    p.top3_volume / NULLIF(t.total_volume, 0) AS wallet_concentration,
    n12.vol_12h   / NULLIF(t.total_volume, 0) AS new_wallet_ratio,
    n6.vol_6h     / NULLIF(t.total_volume, 0) AS new_wallet_ratio_6h,
    b.burst_score,
    d.directional_consensus,
    t.trade_vpin
FROM totals t
CROSS JOIN top3 p
CROSS JOIN new_12h n12
CROSS JOIN new_6h n6
CROSS JOIN burst b
CROSS JOIN directional d
"""

dune_results = {}
for name, config in LABELED_MARKET_CONFIGS.items():
    print(f"\n── {name.upper()} ({config['label']}) ─────────────────")
    try:
        sql = build_labeled_sql(
            config["question_filter"], config["start"],
            config["end"], config["resolution"]
        )
        exec_id = execute_sql(sql)
        print(f"  Execution ID: {exec_id}")
        if poll_until_done(exec_id):
            df = fetch_results(exec_id)
            dune_results[name] = df
            print(f"  ✅ Got {len(df)} row(s)")
            if len(df) > 0:
                print(df.to_string())
        else:
            dune_results[name] = pd.DataFrame()
    except Exception as e:
        print(f"  ❌ Error: {e}")
        dune_results[name] = pd.DataFrame()

print("\n── Summary ──────────────────────────────────")
for name, df in dune_results.items():
    print(f"  {name}: {len(df)} row(s)")
print("\nRun Cell 15 to save dune_results to Drive.")


── MADURO (SUSPECTED) ─────────────────
  ❌ Error: 402 Client Error: Payment Required for url: https://api.dune.com/api/v1/sql/execute

── ZACHXBT (SUSPECTED) ─────────────────
  ❌ Error: 402 Client Error: Payment Required for url: https://api.dune.com/api/v1/sql/execute

── NOBEL (CONFIRMED) ─────────────────
  ❌ Error: 402 Client Error: Payment Required for url: https://api.dune.com/api/v1/sql/execute

── IRAN (CONFIRMED) ─────────────────
  ❌ Error: 402 Client Error: Payment Required for url: https://api.dune.com/api/v1/sql/execute

── Summary ──────────────────────────────────
  maduro: 0 row(s)
  zachxbt: 0 row(s)
  nobel: 0 row(s)
  iran: 0 row(s)

Run Cell 15 to save dune_results to Drive.


In [23]:
#@title Cell 10 — Compute wallet features for labeled markets
# Computes wallet-level features from raw trade data in dune_results.
# Requires dune_results populated from Drive or Cell 9.
# ⏱ ~5 seconds · 0 credits

MARKET_LABELS = {"maduro": "SUSPECTED", "zachxbt": "SUSPECTED", "nobel": "CONFIRMED", "iran": "CONFIRMED"}

def compute_wallet_features(df, resolution_time_str):
    if len(df) == 0: return None
    df = df.copy()
    df["block_time"] = pd.to_datetime(df["block_time"], utc=True)
    df["amount"]     = pd.to_numeric(df["amount"], errors="coerce").fillna(0)
    df["price"]      = pd.to_numeric(df["price"],  errors="coerce").fillna(0)
    df["wallet"]     = df["maker"]

    res_time         = pd.Timestamp(resolution_time_str, tz="UTC")
    total_volume     = df["amount"].sum()
    wallet_vol       = df.groupby("wallet")["amount"].sum().sort_values(ascending=False)
    first_trade      = df.groupby("wallet")["block_time"].min()

    top3_volume          = wallet_vol.head(3).sum()
    wallet_concentration = top3_volume / total_volume if total_volume > 0 else 0

    for hours, key in [(12, "new_wallet_ratio"), (6, "new_wallet_ratio_6h")]:
        new_w = first_trade[first_trade >= res_time - pd.Timedelta(hours=hours)].index
        vol   = df[df["wallet"].isin(new_w)]["amount"].sum()
        locals()[key] = vol / total_volume if total_volume > 0 else 0
    new_wallet_ratio    = df[df["wallet"].isin(first_trade[first_trade >= res_time - pd.Timedelta(hours=12)].index)]["amount"].sum() / total_volume if total_volume > 0 else 0
    new_wallet_ratio_6h = df[df["wallet"].isin(first_trade[first_trade >= res_time - pd.Timedelta(hours=6)].index)]["amount"].sum() / total_volume if total_volume > 0 else 0

    df["hour_bucket"]  = df["block_time"].dt.floor("h")
    burst_score        = df.groupby(["wallet", "hour_bucket"]).size().max() if len(df) > 0 else 0
    outcome_volumes    = df.groupby("token_outcome_name")["amount"].sum()
    dir_consensus      = outcome_volumes.max() / total_volume if total_volume > 0 else 0
    df_s = df.sort_values("block_time")
    prices = df_s["price"].values; amounts = df_s["amount"].values
    buy_v  = amounts[prices > 0.5].sum(); sell_v = amounts[prices <= 0.5].sum()
    trade_vpin = abs(buy_v - sell_v) / (buy_v + sell_v) if (buy_v + sell_v) > 0 else 0

    return {
        "total_volume": total_volume, "unique_wallets": df["wallet"].nunique(),
        "wallet_concentration": wallet_concentration,
        "new_wallet_ratio": new_wallet_ratio, "new_wallet_ratio_6h": new_wallet_ratio_6h,
        "burst_score": int(burst_score), "directional_consensus": dir_consensus,
        "trade_vpin": trade_vpin, "top3_wallets": list(wallet_vol.head(3).index),
        "top3_volume_usd": round(top3_volume, 2),
    }

print("── Wallet-level features (labeled markets) ──────────────────────\n")
wallet_features = {}
for name, df in dune_results.items():
    if len(df) == 0: print(f"⚠️  {name}: no data\n"); continue
    f = compute_wallet_features(df, RESOLUTION_TIMES[name])
    if f is None: continue
    wallet_features[name] = f
    label = MARKET_LABELS[name]
    print(f"── {name.upper()} ({label}) ────────────────────────────")
    print(f"  Total volume:           ${f['total_volume']:,.2f}")
    print(f"  New wallet ratio 12h:   {f['new_wallet_ratio']:.1%}")
    print(f"  New wallet ratio  6h:   {f['new_wallet_ratio_6h']:.1%}")
    print(f"  Burst score:            {f['burst_score']}")
    print(f"  Directional consensus:  {f['directional_consensus']:.1%}")
    print(f"  Trade VPIN:             {f['trade_vpin']:.3f}")
    print()

── Wallet-level features (labeled markets) ──────────────────────

⚠️  maduro: no data

⚠️  zachxbt: no data

⚠️  nobel: no data

⚠️  iran: no data



In [ ]:
#@title Cell 11 — Define wallet suspicion scoring rules
# Rule-based scoring function calibrated from labeled market analysis.
# Each of 5 signals contributes up to 0.20 (max total = 1.0).
# Required by Cell 12 and Cell 13 — always run this cell.
# ⏱ Instant · 0 credits

def compute_wallet_suspicion_score(features):
    """
    Rule-based wallet suspicion score (0.0 – 1.0).

    Thresholds calibrated from confirmed cases:
      Nobel (CONFIRMED):   new_wallet_ratio_12h = 79.8%  — dominant signal
      Maduro (SUSPECTED):  trade_vpin = 0.937            — dominant signal
      ZachXBT (SUSPECTED): burst_score = 270             — dominant signal
      Iran (CONFIRMED):    burst_score = 409, dir_consensus = 77.6%
    """
    score = 0.0
    nwr = features.get("new_wallet_ratio", 0)
    if nwr > 0.50: score += 0.20
    elif nwr > 0.10: score += 0.10

    nwr_6h = features.get("new_wallet_ratio_6h", 0)
    if nwr_6h > 0.10: score += 0.20
    elif nwr_6h > 0.05: score += 0.10

    vpin = features.get("trade_vpin", 0)
    if vpin > 0.80: score += 0.20
    elif vpin > 0.60: score += 0.10

    burst = features.get("burst_score", 0)
    if burst > 200: score += 0.20
    elif burst > 50: score += 0.10

    dc = features.get("directional_consensus", 0)
    if dc > 0.70: score += 0.20
    elif dc > 0.55: score += 0.10

    return round(score, 2)

print("✅ compute_wallet_suspicion_score() defined")
print("   Thresholds: new_wallet_12h >50%=+0.20 >10%=+0.10")
print("              new_wallet_6h  >10%=+0.20  >5%=+0.10")
print("              trade_vpin    >0.80=+0.20 >0.60=+0.10")
print("              burst_score    >200=+0.20  >50=+0.10")
print("              dir_consensus  >70%=+0.20  >55%=+0.10")

✅ compute_wallet_suspicion_score() defined
   Thresholds: new_wallet_12h >50%=+0.20 >10%=+0.10
              new_wallet_6h  >10%=+0.20  >5%=+0.10
              trade_vpin    >0.80=+0.20 >0.60=+0.10
              burst_score    >200=+0.20  >50=+0.10
              dir_consensus  >70%=+0.20  >55%=+0.10


In [ ]:
#@title Cell 12 — Score labeled markets (validation)
# Applies wallet scoring rules to ground truth cases.
# Useful for checking if the model would have caught known insider trading.
# ⏱ Instant · 0 credits

print("── Wallet suspicion scores (labeled markets) ────────────────────\n")
for name, features in wallet_features.items():
    ws = compute_wallet_suspicion_score(features)
    print(f"  {name.upper()} ({MARKET_LABELS[name]}): {ws:.2f}")

print("\n── Dominant signal per confirmed case ────────────────────────────")
for name, note in [
    ("nobel",   "new_wallet_ratio_12h = 79.8%"),
    ("iran",    "burst_score = 409, directional_consensus = 77.6%"),
    ("zachxbt", "burst_score = 270"),
    ("maduro",  "trade_vpin = 0.937"),
]:
    if name in wallet_features:
        print(f"  {name.upper()}: {note}")

── Wallet suspicion scores (labeled markets) ────────────────────


── Dominant signal per confirmed case ────────────────────────────


In [ ]:
#@title Cell 13 — Aggregated Dune query for top N markets [EXPENSIVE]
# Runs a single SQL query returning 1 aggregated row per market (no row limit issues).
#
# COSTS ~4 DUNE CREDITS. Re-run when:
#   - Top 15 changed significantly after a price score refresh
#   - df_wallet_agg is None (not on Drive)
#   - Refreshing the dashboard with fresh wallet data
#
# Skip this cell if df_wallet_agg loaded from Drive ✅ → jump to Cell 14.

top_n = df_scored.nlargest(TOP_N_MARKETS, "suspicion_score").copy().reset_index(drop=True)

def sql_quote(s): return "'" + s.replace("'", "''") + "'"
in_clause = ",\n    ".join(sql_quote(q) for q in top_n["question"].tolist())

aggregated_sql = f"""
WITH trades AS (
    SELECT question, block_time, maker, price, amount, token_outcome_name
    FROM polymarket_polygon.market_trades
    WHERE question IN ({in_clause})
),
market_stats AS (
    SELECT question, COUNT(*) AS trade_count, COUNT(DISTINCT maker) AS unique_wallets,
           SUM(amount) AS total_volume, MAX(block_time) AS last_trade_time, MIN(block_time) AS first_trade_time
    FROM trades GROUP BY question
),
wallet_volumes AS (SELECT question, maker, SUM(amount) AS wallet_volume FROM trades GROUP BY question, maker),
top3 AS (
    SELECT question, SUM(wallet_volume) AS top3_volume
    FROM (SELECT question, wallet_volume, ROW_NUMBER() OVER (PARTITION BY question ORDER BY wallet_volume DESC) AS rn FROM wallet_volumes) r
    WHERE rn <= 3 GROUP BY question
),
resolution_times AS (SELECT question, MAX(block_time) AS inferred_resolution FROM trades GROUP BY question),
new_wallets_12h AS (
    SELECT t.question, SUM(t.amount) AS new_wallet_volume_12h FROM trades t
    JOIN resolution_times r ON t.question = r.question
    WHERE t.maker IN (
        SELECT maker FROM (SELECT maker, question, MIN(block_time) AS first_seen FROM trades GROUP BY maker, question) fw
        JOIN resolution_times rt ON fw.question = rt.question
        WHERE fw.first_seen >= rt.inferred_resolution - INTERVAL '12' HOUR AND fw.question = t.question
    ) AND t.question = r.question GROUP BY t.question
),
new_wallets_6h AS (
    SELECT t.question, SUM(t.amount) AS new_wallet_volume_6h FROM trades t
    JOIN resolution_times r ON t.question = r.question
    WHERE t.maker IN (
        SELECT maker FROM (SELECT maker, question, MIN(block_time) AS first_seen FROM trades GROUP BY maker, question) fw
        JOIN resolution_times rt ON fw.question = rt.question
        WHERE fw.first_seen >= rt.inferred_resolution - INTERVAL '6' HOUR AND fw.question = t.question
    ) AND t.question = r.question GROUP BY t.question
),
burst AS (
    SELECT question, MAX(trades_in_hour) AS burst_score
    FROM (SELECT question, maker, DATE_TRUNC('hour', block_time) AS h, COUNT(*) AS trades_in_hour FROM trades GROUP BY question, maker, DATE_TRUNC('hour', block_time)) x
    GROUP BY question
),
directional AS (
    SELECT question, MAX(ov) * 1.0 / NULLIF(SUM(ov), 0) AS directional_consensus
    FROM (SELECT question, token_outcome_name, SUM(amount) AS ov FROM trades GROUP BY question, token_outcome_name) x
    GROUP BY question
),
vpin AS (
    SELECT question, ABS(yes_vol - no_vol) / NULLIF(yes_vol + no_vol, 0) AS trade_vpin
    FROM (SELECT question, SUM(CASE WHEN price > 0.5 THEN amount ELSE 0 END) AS yes_vol, SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END) AS no_vol FROM trades GROUP BY question) x
)
SELECT ms.question, ms.trade_count, ms.unique_wallets, ms.total_volume, ms.first_trade_time, ms.last_trade_time,
    t3.top3_volume, t3.top3_volume / NULLIF(ms.total_volume, 0) AS wallet_concentration,
    COALESCE(nw12.new_wallet_volume_12h, 0) / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_12h,
    COALESCE(nw6.new_wallet_volume_6h, 0) / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_6h,
    b.burst_score, d.directional_consensus, v.trade_vpin
FROM market_stats ms
LEFT JOIN top3 t3 ON ms.question = t3.question
LEFT JOIN new_wallets_12h nw12 ON ms.question = nw12.question
LEFT JOIN new_wallets_6h nw6 ON ms.question = nw6.question
LEFT JOIN burst b ON ms.question = b.question
LEFT JOIN directional d ON ms.question = d.question
LEFT JOIN vpin v ON ms.question = v.question
ORDER BY ms.question
"""

print(f"── Submitting aggregated Dune query ({len(top_n)} markets) ──────────────")
try:
    exec_id = execute_sql(aggregated_sql)
    print(f"  Execution ID: {exec_id}")
    if poll_until_done(exec_id, timeout=180):
        df_wallet_agg = fetch_results(exec_id)
        print(f"  ✅ Got {len(df_wallet_agg)} rows")
        print(df_wallet_agg[["question", "unique_wallets", "new_wallet_ratio_12h", "burst_score", "trade_vpin"]].to_string())
    else:
        df_wallet_agg = pd.DataFrame()
except Exception as e:
    print(f"  ❌ Error: {e}")
    df_wallet_agg = pd.DataFrame()

── Submitting aggregated Dune query (15 markets) ──────────────
  Execution ID: 01KK9RRYWWGVHCEZZ9CPPEF1FW
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  ✅ Got 15 rows
                                               question  unique_wallets  new_wallet_ratio_12h  burst_score  trade_vpin
0        Hyperlend FDV above $20M one day after launch?            1031          3.610295e-01           53    0.528181
1           Lighter FDV above $3B one day after launch?             638          1.744919e-01           53    0.731299
2         Over $18M committed to the Space public sale?             298          4.647246e-01           20    0.674946
3        Over $1M committed to the Foresee public sale?             182          3.998906e-01           11    0.811

In [ ]:
#@title Cell 14 — Combine price + wallet scores
# Averages price-based (Cell 6) and wallet-based (Cell 13) scores.
# Requires: df_scored, df_wallet_agg, compute_wallet_suspicion_score.
# ⏱ Instant · 0 credits

top_n = df_scored.nlargest(TOP_N_MARKETS, "suspicion_score").copy().reset_index(drop=True)

print(f"── Combined scores (price + wallet) ────────────────────────────")
print(f"{'#':<3} {'Market':<50} {'Price':>7} {'Wallet':>7} {'Combined':>9}")
print("─" * 80)

combined = []
for i, row in top_n.iterrows():
    q           = row["question"]
    price_score = row["suspicion_score"]
    agg = df_wallet_agg[df_wallet_agg["question"] == q] if df_wallet_agg is not None and len(df_wallet_agg) > 0 else pd.DataFrame()

    if len(agg) > 0:
        a = agg.iloc[0]
        wallet_score = compute_wallet_suspicion_score({
            "new_wallet_ratio":      float(a["new_wallet_ratio_12h"] or 0),
            "new_wallet_ratio_6h":   float(a["new_wallet_ratio_6h"]  or 0),
            "trade_vpin":            float(a["trade_vpin"]            or 0),
            "burst_score":           float(a["burst_score"]           or 0),
            "directional_consensus": float(a["directional_consensus"] or 0),
        })
    else:
        wallet_score = None

    combined_score = (price_score + wallet_score) / 2 if wallet_score is not None else price_score
    combined.append({"question": q, "price_score": price_score, "wallet_score": wallet_score, "combined_score": combined_score})
    ws_str = f"{wallet_score:.3f}" if wallet_score is not None else "  N/A"
    print(f"{i+1:<3} {q[:50]:<50} {price_score:>7.3f} {ws_str:>7} {combined_score:>9.3f}")

df_combined = pd.DataFrame(combined).sort_values("combined_score", ascending=False).reset_index(drop=True)
print(f"\n── Top 5 by combined score ──────────────────────────────────────")
print(df_combined[["question", "price_score", "wallet_score", "combined_score"]].head().to_string())

── Combined scores (price + wallet) ────────────────────────────
#   Market                                               Price  Wallet  Combined
────────────────────────────────────────────────────────────────────────────────
1   Sentient FDV above $800M one day after launch?       0.119   0.400     0.259
2   Will Almanak FDV above $50M one day after launch?    0.111   0.300     0.205
3   Over $500K committed to the Sol Raffle public sale   0.085   0.600     0.342
4   Over $20M committed to the Space public sale?        0.079   0.600     0.339
5   Hyperlend FDV above $20M one day after launch?       0.063   0.600     0.332
6   Rainbow FDV above $30M one day after launch?         0.048   0.800     0.424
7   Trove FDV above $5M one day after launch?            0.043   0.500     0.271
8   Over $3M committed to the Infinex public sale?       0.028   0.300     0.164
9   Over $2M committed to the Infinex public sale?       0.013   0.400     0.207
10  Over $7M committed to the Fabric public 

---
## Section D: Save & Export

**Always run both cells** at the end of a session.

- **Cell 15** saves pkl checkpoints to Google Drive — lets you skip re-fetches next session
- **Cell 16** pushes CSVs to GitHub so the [dashboard](https://polymarket-dashboard-roan.vercel.app) updates

In [ ]:
#@title Cell 15 — Save all data to Google Drive
# Always run at end of session to preserve work.
# dune_results is the large file (~28MB); others are small.
# ⏱ ~30 seconds · 0 credits

import pickle, os

SAVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_checkpoint(obj, name):
    if obj is None:
        print(f"  ⚠️  Skipping {name} (is None)")
        return
    path = f"{SAVE_DIR}/{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  ✅ Saved {name} ({os.path.getsize(path)/1024:.1f} KB)")

print(f"Saving to {SAVE_DIR}...\n")
save_checkpoint(df_markets,    "df_markets")
save_checkpoint(df_scored,     "df_scored")
save_checkpoint(histories,     "histories")
save_checkpoint(dune_results,  "dune_results")
save_checkpoint(df_wallet_agg, "df_wallet_agg")
save_checkpoint(df_combined,   "df_combined")

print(f"\nFiles in {SAVE_DIR}:")
for f in sorted(os.listdir(SAVE_DIR)):
    print(f"  {f} ({os.path.getsize(f'{SAVE_DIR}/{f}')/1024:.1f} KB)")

Saving to /content/drive/MyDrive/insider_trading_poc...

  ✅ Saved df_markets (44.8 KB)
  ✅ Saved df_scored (18.4 KB)
  ✅ Saved histories (157.1 KB)
  ✅ Saved dune_results (0.7 KB)
  ✅ Saved df_wallet_agg (3.9 KB)
  ✅ Saved df_combined (1.8 KB)

Files in /content/drive/MyDrive/insider_trading_poc:
  Polymarket Insider Trading Detector — Short Summary.gdoc (0.2 KB)
  df_combined.pkl (1.8 KB)
  df_markets.pkl (44.8 KB)
  df_scored.pkl (18.4 KB)
  df_wallet_agg.pkl (3.9 KB)
  dune_results.pkl (0.7 KB)
  histories.pkl (157.1 KB)


In [ ]:
#@title Cell 16 — Export results to GitHub
# Pushes df_scored, df_wallet_agg, df_combined as CSVs to outputs/ in your GitHub repo.
# Dashboard reads df_combined.csv via GitHub raw URL.
# If file content is unchanged, no new commit is created — that is not an error.
# ⏱ ~10 seconds · 0 credits

from github import Auth, Github
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
g    = Github(auth=Auth.Token(GITHUB_TOKEN))
repo = g.get_repo(GITHUB_REPO)

def push_csv(df, filename, commit_message):
    csv_content = df.to_csv(index=False)
    path = f"outputs/{filename}"
    try:
        existing = repo.get_contents(path, ref=GITHUB_BRANCH)
        repo.update_file(path, commit_message, csv_content, existing.sha, branch=GITHUB_BRANCH)
        print(f"✅ Updated {path}")
    except Exception:
        repo.create_file(path, commit_message, csv_content, branch=GITHUB_BRANCH)
        print(f"✅ Created {path}")

push_csv(df_scored,     "df_scored.csv",     "Update df_scored")
push_csv(df_wallet_agg, "df_wallet_agg.csv", "Update df_wallet_agg")
push_csv(df_combined,   "df_combined.csv",   "Update df_combined")

print(f"\nDone — outputs at https://github.com/{GITHUB_REPO}/tree/{GITHUB_BRANCH}/outputs")

✅ Updated outputs/df_scored.csv
✅ Updated outputs/df_wallet_agg.csv
✅ Updated outputs/df_combined.csv

Done — outputs at https://github.com/chadallen/insider_trading_detection/tree/main/outputs


# Debug

In [13]:
test_sql = """
SELECT COUNT(*) AS n
FROM polymarket_polygon.market_trades
WHERE question LIKE '%Maduro%'
  AND block_time BETWEEN TIMESTAMP '2026-01-28' AND TIMESTAMP '2026-01-31'
"""

exec_id = execute_sql(test_sql)
print(f"Execution ID: {exec_id}")
if poll_until_done(exec_id):
    df = fetch_results(exec_id)
    print(df)

Execution ID: 01KKA1S4RSMSRD3FS453BCMDA7
  Status: QUERY_STATE_EXECUTING — waiting 5s...
      n
0  8155


In [14]:
import requests

exec_id = "01KKA1NY9VXNPDBX9JEHBCEGJ2"
r = requests.get(
    f"https://api.dune.com/api/v1/execution/{exec_id}/status",
    headers=DUNE_HEADERS
)
print(r.json())

{'execution_id': '01KKA1NY9VXNPDBX9JEHBCEGJ2', 'query_id': 0, 'is_execution_finished': True, 'state': 'QUERY_STATE_FAILED', 'submitted_at': '2026-03-09T19:37:43.484637Z', 'expires_at': '2026-06-07T19:37:43.733464Z', 'execution_cost_credits': 0, 'error': {'type': 'FAILED_TYPE_EXECUTION_FAILED', 'message': "line 36:28: '2026-01-31T00:00:00' is not a valid TIMESTAMP literal at line 36, position 28 [Execution ID: 01KKA1NY9VXNPDBX9JEHBCEGJ2]", 'metadata': {'line': 36, 'column': 28}}}
